<a href="https://colab.research.google.com" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import plotly.express as px

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('data/bank_transactions_data_2_augmented_clean_2.csv')
df.columns = df.columns.str.strip()
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'], format='mixed')
df['Month'] = df['TransactionDate'].dt.month
df['Hour'] = df['TransactionDate'].dt.hour
df.fillna(0, inplace=True)

In [ ]:
df

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,Month,Hour
0,TX000001,AC00128,14.09,2023-04-11 16:29:00,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,4,16
1,TX000002,AC00455,376.24,2023-06-27 16:44:00,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,6,16
2,TX000003,AC00019,126.29,2023-07-10 18:16:00,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,7,18
3,TX000004,AC00070,184.50,2023-05-05 16:32:00,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,5,16
4,TX000005,AC00411,13.45,2023-10-16 17:51:00,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,10,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,TX049996,AC00314,69.23,2025-01-28 00:00:00,Debit,Las Vegas,D000546,44.67.137.125,M097,Online,69,Doctor,69,1,6020.29,1,0
49996,TX049997,AC00370,514.53,2022-01-23 00:00:00,Debit,Houston,D000589,140.212.253.222,M061,ATM,46,Engineer,143,1,6371.51,1,0
49997,TX049998,AC00277,118.39,2022-11-08 00:00:00,Debit,Omaha,D000217,152.140.239.181,M029,Online,33,Doctor,296,1,749.34,11,0
49998,TX049999,AC00007,446.99,2025-04-20 00:00:00,Debit,Las Vegas,D000327,131.41.45.13,M082,ATM,58,Doctor,11,1,10915.11,4,0


In [ ]:
df.fillna(0, inplace=True)

In [ ]:
# Saldo vs Monto — tamaño proporcional a intentos de login
fig = px.scatter(
    df.sample(5000, random_state=42),
    x='AccountBalance',
    y='TransactionAmount',
    size='LoginAttempts',
    color='Channel',
    hover_name='CustomerOccupation',
    hover_data=['CustomerAge', 'Location'],
    title='Saldo vs Monto de Transaccion por Canal'
)
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# Distribucion de montos por tipo
fig = px.histogram(
    df,
    x='TransactionAmount',
    nbins=50,
    color='TransactionType',
    barmode='overlay',
    title='Distribucion de Montos por Tipo de Transaccion'
)
fig.show()

In [ ]:
# Promedio de monto por canal
channel_pts = df.groupby('Channel')['TransactionAmount'].mean().reset_index()

fig = px.bar(
    channel_pts,
    x='Channel',
    y='TransactionAmount',
    title='Promedio de Monto por Canal'
)
fig.show()

In [ ]:
# Volumen mensual por canal
monthly = df.groupby(['Month', 'Channel'])['TransactionAmount'].sum().reset_index()

fig = px.line(
    monthly,
    x='Month',
    y='TransactionAmount',
    color='Channel',
    title='Volumen Mensual por Canal',
    markers=True
)
fig.show()

In [ ]:
# Distribucion de montos por ocupacion
fig = px.box(
    df,
    x='CustomerOccupation',
    y='TransactionAmount',
    color='CustomerOccupation',
    title='Distribucion de Montos por Ocupacion del Cliente'
)
fig.show()

In [ ]:
# Transacciones sospechosas por hora (login >= 3)
susp = df[df['LoginAttempts'] >= 3].copy()
susp_hour = susp.groupby('Hour').size().reset_index(name='count')

fig = px.bar(
    susp_hour,
    x='Hour',
    y='count',
    title='Transacciones Sospechosas por Hora del Dia (Login >= 3)'
)
fig.show()

In [ ]:
# Top 15 ciudades interactivo
top_cities = df.groupby('Location')['TransactionAmount'].sum().reset_index()
top_cities = top_cities.sort_values('TransactionAmount', ascending=False).head(15)

fig = px.bar(
    top_cities,
    x='TransactionAmount',
    y='Location',
    orientation='h',
    title='Top 15 Ciudades por Volumen Transaccionado'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()